In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
from dotenv import load_dotenv
import os

load_dotenv()

HOST = os.getenv("HOST")
DATABASE = os.getenv("DATABASE")
USER = os.getenv("USER")
PASSWORD = os.getenv("PASSWORD")
PORT = os.getenv("PORT")

# Bỏ qua cảnh báo của pandas khi dùng psycopg2 trực tiếp với read_sql
warnings.filterwarnings('ignore', category=UserWarning)

try:
    # 1. Kết nối và lấy dữ liệu
    conn = psycopg2.connect(
        host=HOST,
        database=DATABASE,
        user=USER,
        password=PASSWORD,
        port=PORT  # mặc định là 5432
    )
    
    print("Kết nối database thành công!")

    query = """SELECT * FROM "stroke_data";"""
    df = pd.read_sql(query, con=conn)
    print(f"Đã lấy dữ liệu thành công! Kích thước dữ liệu: {df.shape}")

except Exception as e:
    print("Lỗi khi kết nối hoặc lấy dữ liệu:", e)
    df = None # Gán None nếu lỗi để tránh lỗi ở phần sau

finally:
    # Đảm bảo đóng kết nối sau khi lấy xong data
    if 'conn' in locals() and conn:
        conn.close()
        print("Đã đóng kết nối database an toàn.\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mtick

# Lọc chỉ giữ Male và Female
df_filtered = df[df['gender'].isin(['Male', 'Female', 'Other'])].copy()

genders = ['Male', 'Female', 'Other']              
gender_colors = {'Female': '#F8766D', 'Male': '#00BFC4', 'Other': "#0C007A"}     

stroke_labels = [0, 1]
stroke_names = {0: 'No Stroke', 1: 'Stroke'}
hypertension_levels = [0, 1]              

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

for ax, stroke_val in zip(axes, stroke_labels):
    counts = {(h, g): 0 for h in hypertension_levels for g in genders}

    sub = df_filtered[df_filtered['stroke'] == stroke_val]
    grp = sub.groupby(['hypertension', 'gender']).size()

    for (h, g), c in grp.items():
        if h in hypertension_levels and g in genders:
            counts[(h, g)] = int(c)

    # 📍 [THAY ĐỔI QUAN TRỌNG NHẤT Ở ĐÂY]
    # Tính tổng số người trong toàn bộ NHÓM ĐỘT QUỴ hiện tại (Stroke = 0 hoặc Stroke = 1)
    # Tổng của nhóm này sẽ được coi là 100%
    total_in_stroke_group = len(sub)

    x = np.arange(len(hypertension_levels))
    width = 0.8
    bottom = np.zeros(len(hypertension_levels))

    for g in genders:
        # 📍 Chia số lượng của từng nhóm nhỏ cho TỔNG SỐ NGƯỜI TRONG NHÓM ĐỘT QUỴ
        heights = [
            (counts[(h, g)] / total_in_stroke_group * 100) if total_in_stroke_group > 0 else 0 
            for h in hypertension_levels
        ]
        
        ax.bar(x, heights, width=width, bottom=bottom,
               color=gender_colors[g], label=g)
        bottom += np.array(heights)

    # Format
    ax.set_title(stroke_names[stroke_val], fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([str(h) for h in hypertension_levels])
    ax.set_xlabel('Hypertension')
    ax.set_facecolor('#EBEBEB')           
    ax.grid(True, axis='y', color='white', linewidth=1.2)
    ax.set_axisbelow(True)
    
    # Cố định trục Y từ 0 đến 100%
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# Đổi tên trục Y cho chuẩn ý nghĩa biểu đồ
axes[0].set_ylabel('Percentage of Stroke Group (%)')

fig.suptitle('Proportion of Hypertension and Gender within Stroke Groups',
             fontsize=16, fontweight='bold', y=0.98)

# Legend
handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(), title='gender', loc='center right', frameon=False)

plt.tight_layout(rect=[0, 0, 0.88, 0.95])
plt.show()